In [ ]:
"""
1. Unstructured Pruning — ResNet-50 / CIFAR-10
===============================================
Removes individual weights (fine-grained) across the entire network
using global L1 magnitude ranking. Weights become zero but tensors
stay dense — no structural change to the model.

Output: 1_unstructured_Pruning.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "1è1_unstructured_Pruning.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

SPARSITY_LEVELS  = [0.30, 0.50, 0.70, 0.80, 0.90]
MAX_ACC_DROP     = 0.02
INFERENCE_RUNS   = 100


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path):
    model = build_model(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

# ── Pruning ───────────────────────────────────────────────────────────────────
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]

def apply_pruning(model, sparsity):
    """Global L1 unstructured pruning — removes individual weights."""
    pruned = copy.deepcopy(model)
    layers = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
    for module, param_name in layers:
        prune.remove(module, param_name)
    return pruned

def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0

def count_params(model):
    total  = sum(p.numel() for p in model.parameters())
    active = sum((p != 0).sum().item() for p in model.parameters())
    return total, active

def dense_size_mb(model):
    return sum(p.numel() for p in model.parameters()) * 4 / 1024**2

def sparse_size_mb(model):
    active = sum((p != 0).sum().item() for p in model.parameters())
    return active * 4 / 1024**2

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)
    return size

# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def measure_gpu_ms(model):
    if DEVICE.type != "cuda": return -1.0
    model.eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(20): model(dummy)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(INFERENCE_RUNS): model(dummy)
        e.record()
        torch.cuda.synchronize()
    return s.elapsed_time(e) / INFERENCE_RUNS

def measure_cpu_ms(model):
    cpu_m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(BATCH_SIZE, 3, 32, 32)
    with torch.no_grad():
        for _ in range(5): cpu_m(dummy)
        t0 = time.perf_counter()
        for _ in range(20): cpu_m(dummy)
    return (time.perf_counter() - t0) / 20 * 1000


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  1. Unstructured Pruning — ResNet-50 / CIFAR-10")
    print(f"  Device: {DEVICE}  |  Sparsity: {[int(s*100) for s in SPARSITY_LEVELS]}")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)

    model  = load_baseline(BASELINE_MODEL_PATH)
    loader = get_test_loader()

    base_dense = dense_size_mb(model)
    base_disk  = disk_size_mb(model)

    results       = []
    best_model    = None
    best_score    = -1.0
    best_sparsity = None

    for sparsity in SPARSITY_LEVELS:
        print(f"\n  Pruning at {int(sparsity*100)}%...")
        pruned = apply_pruning(model, sparsity)
        actual_sp = real_sparsity(pruned)
        total_p, active_p = count_params(pruned)

        metrics  = evaluate(pruned, loader)
        inf_gpu  = measure_gpu_ms(pruned)
        inf_cpu  = measure_cpu_ms(pruned)
        acc_drop = baseline["accuracy"] - metrics["accuracy"]

        d_mb  = dense_size_mb(pruned)
        sp_mb = sparse_size_mb(pruned)
        dk_mb = disk_size_mb(pruned)

        pruned_flops, pruned_flops_G = compute_flops(pruned, DEVICE)


        row = {
            "target_sparsity"            : sparsity,
            "actual_sparsity"            : round(actual_sp, 4),
            "accuracy"                   : round(metrics["accuracy"], 6),
            "precision"                  : round(metrics["precision"], 6),
            "recall"                     : round(metrics["recall"], 6),
            "f1"                         : round(metrics["f1"], 6),
            "accuracy_drop"              : round(acc_drop, 6),
            "params_total"               : total_p,
            "params_active"              : active_p,
            "size_dense_mb"              : round(d_mb, 4),
            "size_sparse_theoretical_mb" : round(sp_mb, 4),
            "size_disk_mb"               : round(dk_mb, 4),
            "disk_saved_mb"              : round(base_disk - dk_mb, 4),
            "inference_gpu_ms"           : round(inf_gpu, 4) if inf_gpu > 0 else None,
            "inference_cpu_ms"           : round(inf_cpu, 4),
        }
        results.append(row)

        print(f"  Sparsity: {actual_sp*100:.1f}% | Acc: {metrics['accuracy']:.4f} "
              f"(drop: {acc_drop:+.4f}) | Sparse MB: {sp_mb:.2f}")

        if acc_drop <= MAX_ACC_DROP:
            score = metrics["accuracy"] + ((base_disk - dk_mb) / base_disk) * 0.1
            if score > best_score:
                best_score, best_model, best_sparsity = score, copy.deepcopy(pruned), sparsity

    report = {
        "method"            : "unstructured_pruning",
        "description"       : "Global L1 unstructured pruning — individual weights zeroed",
        "pruning_granularity": "weight",
        "baseline"          : baseline,
        "baseline_size_breakdown": {
            "dense_ram_mb": round(base_dense, 4),
            "disk_pth_mb" : round(base_disk, 4),
        },
        "pruning_criterion" : "L1 magnitude (|w|)",
        "device"            : str(DEVICE),
        "max_acc_drop_threshold": MAX_ACC_DROP,
        "best_sparsity"     : best_sparsity,
        "results"           : results,
        "notes": {
            "size_dense_mb"              : "Full float32 RAM — never shrinks",
            "size_sparse_theoretical_mb" : "active_params * 4B — sparse format lower bound",
            "size_disk_mb"               : "Actual .pth (compressed zip)",
            "gpu_speedup"                : "Dense CUDA does NOT skip zeros — no GPU speedup without cuSPARSE",
        }
    }
    

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved -> {OUTPUT_JSON}")


if __name__ == "__main__":
    main()



  1. Unstructured Pruning — ResNet-50 / CIFAR-10
  Device: cuda  |  Sparsity: [30, 50, 70, 80, 90]


  Pruning at 30%...
  Sparsity: 30.0% | Acc: 0.9321 (drop: -0.0001) | Sparse MB: 62.87

  Pruning at 50%...
  Sparsity: 50.0% | Acc: 0.9320 (drop: +0.0000) | Sparse MB: 44.96

  Pruning at 70%...
  Sparsity: 70.0% | Acc: 0.9319 (drop: +0.0001) | Sparse MB: 27.06

  Pruning at 80%...
  Sparsity: 80.0% | Acc: 0.9276 (drop: +0.0044) | Sparse MB: 18.11

  Pruning at 90%...
  Sparsity: 90.0% | Acc: 0.8774 (drop: +0.0546) | Sparse MB: 9.15

  ✓ Saved -> 3_unstructured_Pruning.json
